# Single-Model Colab Runner

This notebook is a minimal Colab wrapper for the existing curriculum-hacking pipeline.
It is designed for low-compute usage: run one model at a time, store results to Drive,
and compare runs later.


## What This Does

- Uses the existing `train_time_prompt_opt.py` script
- Lets you choose a single base model
- Lets you run on the biased dataset, the unbiased dataset, or both
- Saves outputs to Google Drive
- Keeps the Qwen pipeline logic unchanged; this only swaps `--base-model` and `--train-file`


In [ ]:
%%capture
import os
import subprocess

os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

!pip install --upgrade -qqq uv

try:
    import numpy, PIL
    _numpy = f"numpy=={numpy.__version__}"
    _pil = f"pillow=={PIL.__version__}"
except Exception:
    _numpy = "numpy"
    _pil = "pillow"

try:
    is_t4 = "Tesla T4" in subprocess.check_output(["nvidia-smi"]).decode()
except Exception:
    is_t4 = False

_vllm, _triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.15.1", "triton")

!uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
!uv pip install -qqq {_triton}
!uv pip install -qqq transformers==4.56.2 datasets peft accelerate numpy pillow python-dotenv huggingface_hub pandas
!uv pip install -qqq --no-deps trl==0.22.2
!uv pip uninstall -y -qqq torchcodec || true


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/armaansandhu26/ippo.git"
REPO_DIR = Path('/content/ippo')

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Using existing repo at {REPO_DIR}')

%cd /content/ippo


In [ ]:
# Optional: needed for gated Meta Llama checkpoints.
from huggingface_hub import notebook_login

# notebook_login()


In [ ]:
from pathlib import Path

# Pick one model at a time.
# Examples:
#   'Qwen/Qwen2.5-1.5B-Instruct'
#   'unsloth/Llama-3.2-1B-Instruct-bnb-4bit'
BASE_MODEL = 'unsloth/Llama-3.2-1B-Instruct-bnb-4bit'

# For distribution comparison, condition 0 is the simplest control.
CONDITION = '0'
SEED = 42
BETA = 0.1
CACHE_DIR = '/content/hf-cache'

# Choose one of: 'biased', 'unbiased', 'both'
TRAINING_DISTRIBUTION = 'both'

TRAIN_FILES = {
    'biased': 'data/processed/prelim_train.jsonl',
    'unbiased': 'data/processed/unbiased_prelim_train_with_original_reasoning.jsonl',
}

DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/ippo_outputs')
EXPERIMENT_TAG = 'single_model_distribution_compare'
OUTPUT_BASE = DRIVE_OUTPUT_ROOT / EXPERIMENT_TAG
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

print({
    'base_model': BASE_MODEL,
    'condition': CONDITION,
    'seed': SEED,
    'training_distribution': TRAINING_DISTRIBUTION,
    'output_base': str(OUTPUT_BASE),
})


In [ ]:
# Optional sanity check.
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=1024,
    load_in_4bit=True,
    cache_dir=CACHE_DIR,
)
print('Loaded successfully:', BASE_MODEL)
del model


In [ ]:
import json
import shlex
from pathlib import Path

def selected_distributions(mode: str) -> list[str]:
    if mode == 'biased':
        return ['biased']
    if mode == 'unbiased':
        return ['unbiased']
    if mode == 'both':
        return ['biased', 'unbiased']
    raise ValueError(f'Unknown TRAINING_DISTRIBUTION: {mode}')

model_slug = BASE_MODEL.split('/')[-1]
run_plan = []
for distribution in selected_distributions(TRAINING_DISTRIBUTION):
    output_root = OUTPUT_BASE / distribution / CONDITION / model_slug / f'seed_{SEED}'
    output_root.mkdir(parents=True, exist_ok=True)
    cmd = [
        'python3',
        'scripts/curriculum_hacked/train_time_prompt_opt.py',
        '--condition', CONDITION,
        '--base-model', BASE_MODEL,
        '--train-file', TRAIN_FILES[distribution],
        '--seed', str(SEED),
        '--beta', str(BETA),
        '--cache-dir', CACHE_DIR,
        '--output-root', str(output_root),
    ]
    run_plan.append({
        'distribution': distribution,
        'train_file': TRAIN_FILES[distribution],
        'output_root': str(output_root),
        'cmd': cmd,
    })

print(json.dumps([{k: v for k, v in item.items() if k != 'cmd'} for item in run_plan], indent=2))


In [ ]:
import subprocess

for item in run_plan:
    print('\nRunning:')
    print(' '.join(shlex.quote(part) for part in item['cmd']))
    subprocess.run(item['cmd'], check=True)


In [ ]:
import json
from pathlib import Path
import pandas as pd

rows = []
for item in run_plan:
    train_eval = Path(item['output_root']) / 'final_train_eval.json'
    test_eval = Path(item['output_root']) / 'final_eval.json'
    row = {
        'base_model': BASE_MODEL,
        'distribution': item['distribution'],
        'condition': CONDITION,
        'seed': SEED,
        'train_file': item['train_file'],
        'output_root': item['output_root'],
        'train_accuracy': None,
        'train_a_rate': None,
        'test_accuracy': None,
        'test_a_rate': None,
    }

    if train_eval.exists():
        data = json.loads(train_eval.read_text())
        metrics = data.get('final_eval_metrics', {})
        row['train_accuracy'] = metrics.get('accuracy')
        row['train_a_rate'] = metrics.get('a_rate')

    if test_eval.exists():
        data = json.loads(test_eval.read_text())
        metrics = data.get('final_eval_metrics', {})
        row['test_accuracy'] = metrics.get('accuracy')
        row['test_a_rate'] = metrics.get('a_rate')

    rows.append(row)

df = pd.DataFrame(rows)
display(df)

summary_dir = OUTPUT_BASE / 'stitched_results'
summary_dir.mkdir(parents=True, exist_ok=True)
summary_csv = summary_dir / f'{model_slug}_seed_{SEED}_summary.csv'
summary_json = summary_dir / f'{model_slug}_seed_{SEED}_summary.json'
df.to_csv(summary_csv, index=False)
summary_json.write_text(json.dumps(rows, indent=2))

print('Saved:')
print('-', summary_csv)
print('-', summary_json)


## Recommended First Run

For your immediate Colab experiment:

- `BASE_MODEL = 'unsloth/Llama-3.2-1B-Instruct-bnb-4bit'`
- `CONDITION = '0'`
- `TRAINING_DISTRIBUTION = 'both'`
- `SEED = 42`

That gives you one biased run and one unbiased run for the same model,
stored separately in Drive, with a small stitched summary at the end.
